In [1]:
# --- Colab bootstrap -------------------------------------------------------
import os, sys
if not (os.path.isdir('../thermo') or os.path.isdir('thermo')):
    !rm -rf /tmp/cbet6e
    !git clone -q --depth 1 https://github.com/emfurst/cbet6e.git /tmp/cbet6e
    !cp -r /tmp/cbet6e/code/thermo /tmp/cbet6e/code/data .
    sys.path.insert(0, '.')
# ---------------------------------------------------------------------------


# Bioconcentration in one line, and a field measurement to check it against (Illustration 12.5-2)

PCBs were made as transformer oil. They leaked, they do not degrade, and they are now
everywhere. In the St. Lawrence River the water carries about **0.3 ppb** of them. What is
in the fish?

The whole calculation is Eq. 12.5-9 and a multiplication:

$$K_{{\rm BW},i} = w_{\rm B}K_{{\rm OW},i}, \qquad C^{\rm B} = K_{\rm BW}\,C^{\rm W}$$

with $w_{\rm B} = 0.05$ for the lipid fraction of a fish and
$\log_{10}K_{\rm OW} = 5.5$ for the PCB average. Two numbers, one product.

**And there is a real measurement to check it against**, which is what makes this
illustration worth a notebook rather than a line of arithmetic: field testing found
**7.9 ppm** of PCBs in eels from the St. Lawrence. The prediction is 4.74 ppm. A
thermodynamic model with no fitted parameter, applied to a live animal in a river, lands
within a factor of two.

**The number to take away is not 4.74 ppm, it is 15 800.** The fish concentrates PCBs
from its water by four orders of magnitude, and that factor -- not the concentration in the
water -- is what makes a 0.3 ppb river a public health problem.

SIS is Stanley I. Sandler, *Chemical, Biochemical, and Engineering Thermodynamics*.

Eric Furst
August 2026

In [2]:

import sys; sys.path.append("..")
import numpy as np

from thermo.partition import compartment_partition, COMPARTMENTS

LOG_KOW_PCB = 5.5                  # the average for PCBs, as Illustration 12.5-2 gives it
C_WATER_PPB = 0.3                  # ppb by weight, St. Lawrence River
C_EEL_PPM_MEASURED = 7.9           # ppm by weight, field testing

KOW = 10.0 ** LOG_KOW_PCB
K_BW = float(compartment_partition(KOW, "biota"))
C_B_PPB = K_BW * C_WATER_PPB

print("  Illustration 12.5-2: PCBs in St. Lawrence eels")
print(f"    K_OW  = 10^{LOG_KOW_PCB} = {KOW:.4g}")
print(f"    w_B   = {COMPARTMENTS['biota']['w']}   (lipid fraction of a fish, Sec. 12.5)")
print(f"    K_BW  = w_B K_OW = {K_BW:.4g}          SIS 1.58e4")
print(f"\n    C^B = K_BW C^W = {K_BW:.4g} x {C_WATER_PPB} ppb"
      f" = {C_B_PPB:.0f} ppb = {C_B_PPB / 1000:.2f} ppm")
print(f"    SIS: 4740 ppb = 4.74 ppm")
print(f"\n    measured in eels: {C_EEL_PPM_MEASURED} ppm")
print(f"    predicted / measured = {(C_B_PPB / 1000) / C_EEL_PPM_MEASURED:.2f}"
      f"   -- within a factor of"
      f" {C_EEL_PPM_MEASURED / (C_B_PPB / 1000):.1f}")

  Illustration 12.5-2: PCBs in St. Lawrence eels
    K_OW  = 10^5.5 = 3.162e+05
    w_B   = 0.05   (lipid fraction of a fish, Sec. 12.5)
    K_BW  = w_B K_OW = 1.581e+04          SIS 1.58e4

    C^B = K_BW C^W = 1.581e+04 x 0.3 ppb = 4743 ppb = 4.74 ppm
    SIS: 4740 ppb = 4.74 ppm

    measured in eels: 7.9 ppm
    predicted / measured = 0.60   -- within a factor of 1.7



## The bioconcentration factor, which is the actual result

Section 12.5's Comment puts it as a ratio: the eels carry PCBs at 26 300 times the
concentration in the river water. That is the measured factor. The model's factor is
$K_{\rm BW} = 15\,800$.

**Why a fish concentrates anything at all** is one line of thermodynamics and no biology.
At equilibrium the PCB has the same fugacity in water and in lipid. PCBs are hydrophobic,
so their activity coefficient in water is enormous and in lipid is near one -- and
Eq. 11.4-11 says the partition coefficient *is* that ratio. The fish does not pump PCBs in.
It simply offers them somewhere far more comfortable to sit, and 5 % of its mass is enough.

In [3]:

BCF_MEASURED = C_EEL_PPM_MEASURED * 1000 / C_WATER_PPB

print("  Bioconcentration factors")
print(f"    predicted (= K_BW)   {K_BW:9.0f}")
print(f"    measured             {BCF_MEASURED:9.0f}      SIS 'a factor of 26 300'")
print(f"    ratio                {BCF_MEASURED / K_BW:9.2f}")

print(f"\n  What a single number would have to change to close the gap")
print(f"    {'if this were the only error':<34} {'value needed':>13} {'vs Sec. 12.5':>13}")
print(f"    {'lipid weight fraction w_B':<34}"
      f" {BCF_MEASURED / KOW:13.3f} {COMPARTMENTS['biota']['w']:>13}")
print(f"    {'log10 K_OW of the PCB mixture':<34}"
      f" {np.log10(BCF_MEASURED / COMPARTMENTS['biota']['w']):13.2f}"
      f" {LOG_KOW_PCB:>13}")
d_log = np.log10(BCF_MEASURED / COMPARTMENTS["biota"]["w"]) - LOG_KOW_PCB
print(f"\n    Either an eel that is {BCF_MEASURED / KOW * 100:.0f} % lipid rather than 5 %,")
print(f"    or a PCB mixture {d_log:.2f} log units more hydrophobic than the average.")
print(f"    Both are small. A factor of {BCF_MEASURED / K_BW:.1f} on a bioconcentration")
print(f"    factor of {K_BW:.0f} is covered by a {d_log:.2f} shift in ONE log-scale input,")
print(f"    which is the real result here: the disagreement is inside the uncertainty of")
print(f"    the inputs, so it is not evidence against the model.")
print(f"\n    One mechanism worth checking, and it is a hypothesis rather than a")
print(f"    finding: the illustration's own phrase is 'the AVERAGE octanol-water")
print(f"    partition coefficient for PCBs'. PCB is not a compound but 209 congeners,")
print(f"    and the ones that persist long enough to be found in an eel are the heavily")
print(f"    chlorinated, least degradable end of the distribution -- which is also the")
print(f"    most hydrophobic end. If so the average is the wrong statistic, and it is")
print(f"    wrong in the direction the discrepancy points. Settling that would take the")
print(f"    congener distribution actually measured in those eels, which this")
print(f"    illustration does not give.")

  Bioconcentration factors
    predicted (= K_BW)       15811
    measured                 26333      SIS 'a factor of 26 300'
    ratio                     1.67

  What a single number would have to change to close the gap
    if this were the only error         value needed  vs Sec. 12.5
    lipid weight fraction w_B                  0.083          0.05
    log10 K_OW of the PCB mixture               5.72           5.5

    Either an eel that is 8 % lipid rather than 5 %,
    or a PCB mixture 0.22 log units more hydrophobic than the average.
    Both are small. A factor of 1.7 on a bioconcentration
    factor of 15811 is covered by a 0.22 shift in ONE log-scale input,
    which is the real result here: the disagreement is inside the uncertainty of
    the inputs, so it is not evidence against the model.

    One mechanism worth checking, and it is a hypothesis rather than a
    finding: the illustration's own phrase is 'the AVERAGE octanol-water
    partition coefficient for PCBs'. P


## Your turn

1. Recompute the prediction using $\log_{10}K_{\rm OW}$ for a specific heavily
   chlorinated congener -- say 2,2',4,4',5,5'-hexachlorobiphenyl -- instead of the PCB
   average. How much of the gap closes?
2. Section 12.5 takes $w_{\rm B} = 0.05$ as typical. Eels are notably fatty. Look up a
   lipid fraction for eel specifically and redo the calculation; then say which of the
   two inputs you would rather have measured.
3. This calculation is an *equilibrium* one. A PCB's environmental half-life is over 20
   years and a river flushes in days. Which of those timescales justifies the equilibrium
   assumption, and for which compartment might it fail?
4. Problem 12.5-2 puts four insecticides in four fish tanks at saturation. Compute the
   fish concentration for each and rank them. Then rank them by water solubility. The two
   orderings are nearly reversed -- explain why in terms of Eq. 11.4-11.
5. Using Illustration 12.5-1's route, estimate the air-water partition coefficient for
   PCBs from the vapor pressure in Problem 12.5-1 ($2\times10^{-7}$ bar) and the water
   solubility given there (0.1 mg/L), and say whether a river or the air above it is the
   larger reservoir.